# Training `waste-v1.pt`

This notebook produces the custom detector that **Stage 5** of the project has been
specified for but never had. Run it top to bottom on a free Colab GPU; it takes
roughly 40–90 minutes depending on dataset size and epochs.

### What it fixes

The deployed system runs COCO-pretrained `yolov8n.pt`. COCO has no idea what a
**battery**, an **aluminium can**, **cardboard** or a **plastic bag** is — the four
classes that matter most in municipal waste, and the reason `config/labels_waste.yaml`
exists. A battery in the dry recyclable stream is a fire in a collection vehicle;
that is the single most consequential sorting error this system can make, and today
it cannot even see one.

### What it does NOT change

Nothing above the detector. The perception seam means integration is two lines in
`config/config.yaml`:

```yaml
detection:
  model_path: "models/waste-v1.pt"
  label_map:  "config/labels_waste.yaml"
```

No agent code changes. `tests/test_labels.py` already runs the real agents against
that vocabulary, so the moment the weights exist a battery routes to hazardous.

---
**Before you start:** Runtime → Change runtime type → **T4 GPU**. Training on Colab's
CPU takes more than a day, and the notebook refuses to start one.


## 1 · Confirm the GPU

In [ ]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then Runtime -> Restart session and run this cell again.\n"
        "Training yolov8n on Colab's CPU takes 20+ hours; it is not worth attempting."
    )
print(out.stdout.split("\n")[8] if len(out.stdout.split("\n")) > 8 else out.stdout)
print("GPU present. Continue.")

## 2 · Get the project

Cloning your repo means the notebook uses **your** `training/train.py` and **your**
`config/labels_waste.yaml`, so the class names it trains are checked against the
vocabulary the agents actually read. That check is the whole point: a class the label
map has never heard of trains perfectly well and then reports `UNKNOWN` for every
instance at run time, which surfaces only as an unexplained rise in manual
inspections.

In [ ]:
REPO = "https://github.com/HARSHAL-SEMICOLON/MUNCIPAL-CORP.git"

import os, shutil
if os.path.exists("project"):
    shutil.rmtree("project")
!git clone -q $REPO project
%cd project
!ls

In [ ]:
!pip install -q ultralytics lap roboflow
import ultralytics; ultralytics.checks()

## 3 · Get a labelled dataset

You need images **with bounding boxes**. This is worth being blunt about, because it
is where most waste-classification projects quietly go wrong:

| Dataset | Usable here? |
|---|---|
| **TrashNet** | **No.** It is a *classification* dataset — one label per image, no boxes. It can never train a detector, whatever tutorials claim. |
| **TACO** | Yes, but thin: ~1500 images across 60 classes, so roughly 80 per class. Usable combined with others. |
| **Roboflow Universe** | Best option. Several waste-detection sets with boxes, exportable in YOLOv8 format in one call. |

Sign in free at **roboflow.com**, open a waste-detection dataset on Universe, press
**Download this Dataset → YOLOv8 → show download code**, and paste your snippet below.
Your API key is personal — do not commit this notebook with it filled in.

In [ ]:
# Paste your Roboflow snippet here. Example shape:
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("some-workspace").project("some-waste-project")
# dataset = project.version(3).download("yolov8")

from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_API_KEY")
project = rf.workspace("PASTE_WORKSPACE").project("PASTE_PROJECT")
dataset = project.version(1).download("yolov8")

print("downloaded to:", dataset.location)

## 4 · Remap the dataset's classes onto your vocabulary

**The step that makes the trained model drop straight in.**

A Roboflow set calls things whatever its author chose — `plastic`, `Metal`, `bottle-glass`.
Your `training/dataset.yaml` names ten specific classes, and `config/labels_waste.yaml`
maps exactly those to materials. If the names disagree, every prediction lands as
`UNKNOWN`.

So this cell rewrites the dataset in place: it maps each source class onto one of your
ten, renumbers every label file accordingly, and **drops** boxes whose class has no
sensible home rather than forcing them somewhere wrong. Read the printed summary before
training — that is your last chance to catch a bad mapping cheaply.

In [ ]:
import yaml
from pathlib import Path
from collections import Counter

DATA = Path(dataset.location)

# Your ten, in the order training/dataset.yaml defines them.
TARGET = ["battery", "aluminium_can", "cardboard", "plastic_bag", "pet_bottle",
          "glass_bottle", "paper", "food_waste", "mobile_phone", "plastic_container"]

# Source name (lowercased) -> your class. EDIT THIS to match your dataset after
# looking at the "source classes" print below. Anything unmapped is dropped.
REMAP = {
    "battery": "battery", "batteries": "battery",
    "aluminium_can": "aluminium_can", "aluminum can": "aluminium_can",
    "can": "aluminium_can", "cans": "aluminium_can", "metal can": "aluminium_can",
    "cardboard": "cardboard", "carton": "cardboard",
    "plastic_bag": "plastic_bag", "plastic bag": "plastic_bag", "bag": "plastic_bag",
    "plastic bottle": "pet_bottle", "pet": "pet_bottle", "pet_bottle": "pet_bottle",
    "bottle": "pet_bottle",
    "glass bottle": "glass_bottle", "glass": "glass_bottle", "glass_bottle": "glass_bottle",
    "paper": "paper", "newspaper": "paper",
    "biodegradable": "food_waste", "food": "food_waste", "food_waste": "food_waste",
    "organic": "food_waste",
    "mobile": "mobile_phone", "phone": "mobile_phone", "e-waste": "mobile_phone",
    "plastic": "plastic_container", "plastic_container": "plastic_container",
    "container": "plastic_container",
}

src_yaml = yaml.safe_load((DATA / "data.yaml").read_text())
src_names = [str(n) for n in src_yaml["names"]]
print("source classes:", src_names)
print()

old_to_new = {}
for i, name in enumerate(src_names):
    target = REMAP.get(name.strip().lower())
    old_to_new[i] = TARGET.index(target) if target in TARGET else None
    print(f"  {name:24s} -> {target if target else '*** DROPPED ***'}")

kept, dropped = Counter(), 0
for split in ("train", "valid", "test"):
    label_dir = DATA / split / "labels"
    if not label_dir.exists():
        continue
    for txt in label_dir.glob("*.txt"):
        lines_out = []
        for line in txt.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            new = old_to_new.get(int(parts[0]))
            if new is None:
                dropped += 1
                continue
            kept[TARGET[new]] += 1
            lines_out.append(" ".join([str(new)] + parts[1:]))
        txt.write_text("\n".join(lines_out) + ("\n" if lines_out else ""))

print()
print("boxes kept per class:")
for name in TARGET:
    n = kept.get(name, 0)
    flag = "  <-- too few to learn" if 0 < n < 100 else ("  <-- ABSENT" if n == 0 else "")
    print(f"  {name:20s} {n:6d}{flag}")
print(f"\nboxes dropped (unmapped): {dropped}")

**Read that table before continuing.**

A class with **0** boxes cannot be learned and should come out of `dataset.yaml`.
A class with **under ~100** produces a model that is *confidently wrong* about it —
which is precisely the failure this project's architecture exists to avoid. Leaving a
class out is better than including it with thirty examples: the system reports `UNKNOWN`
and asks a human, which is the honest outcome.

If several classes are thin, either find a second dataset and merge, or cut
`dataset.yaml` down to the classes you actually have.

## 5 · Point the dataset file at the downloaded data

In [ ]:
present = [n for n in TARGET if kept.get(n, 0) > 0]
print("training on:", present)

names = {i: n for i, n in enumerate(present)}
# Renumber again if any class was cut, so indices stay contiguous.
if present != TARGET:
    reindex = {TARGET.index(n): i for i, n in enumerate(present)}
    for split in ("train", "valid", "test"):
        d = DATA / split / "labels"
        if not d.exists():
            continue
        for txt in d.glob("*.txt"):
            out = []
            for line in txt.read_text().splitlines():
                p = line.split()
                if p and int(p[0]) in reindex:
                    out.append(" ".join([str(reindex[int(p[0])])] + p[1:]))
            txt.write_text("\n".join(out) + ("\n" if out else ""))

spec = {
    "path": str(DATA),
    "train": "train/images",
    "val": "valid/images" if (DATA / "valid").exists() else "train/images",
    "names": names,
}
Path("training/dataset_colab.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
print(Path("training/dataset_colab.yaml").read_text())

## 6 · Check the class names against the label map

This runs **your** `training/train.py --check`. It validates that every class name in
the dataset exists in `config/labels_waste.yaml`. If it complains, fix the mapping in
step 4 rather than pressing on — a mismatch here is invisible until run time.

In [ ]:
!python -m training.train --data training/dataset_colab.yaml --check

## 7 · Train

`yolov8n` is the right size deliberately: it is what a laptop can run at frame rate and
what a Phase 2 microcontroller-driven line could plausibly be fed by. A bigger model
would score better on paper and miss the point.

100 epochs on a T4 takes roughly 40–90 minutes for a few thousand images. Start with 60
if you want a result sooner; you can always resume.

In [ ]:
!python -m training.train \
    --data training/dataset_colab.yaml \
    --model yolov8n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --name waste-v1

## 8 · Read the results honestly

`mAP50` is the headline, but the **per-class** table is what matters for this project.
A high average hiding a battery class at 0.2 is a worse outcome than a lower average
with battery at 0.8 — the hazardous stream is the one where an error costs most.

Look for classes far below the rest: those are the ones to gather more images of, and
the ones to consider cutting until you have.

In [ ]:
from pathlib import Path
import glob

run = sorted(glob.glob("runs/detect/waste-v1*"))[-1]
print("run:", run)

from IPython.display import Image, display
for img in ["results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"]:
    p = Path(run) / img
    if p.exists():
        print(f"\n=== {img} ===")
        display(Image(filename=str(p), width=900))

In [ ]:
# Per-class numbers on the validation split.
from ultralytics import YOLO
best = f"{run}/weights/best.pt"
metrics = YOLO(best).val(data="training/dataset_colab.yaml", split="val")
print()
print(f"{'class':22s} {'mAP50':>8s} {'mAP50-95':>10s}")
for i, name in metrics.names.items():
    try:
        print(f"{name:22s} {metrics.box.ap50[i]:8.3f} {metrics.box.ap[i]:10.3f}")
    except (IndexError, TypeError):
        pass
print(f"\n{'OVERALL':22s} {metrics.box.map50:8.3f} {metrics.box.map:10.3f}")

## 9 · Download the weights

Save `waste-v1.pt` into your project's `models/` folder (it is gitignored, which is
correct — binaries do not belong in git).

In [ ]:
import shutil
shutil.copy(best, "waste-v1.pt")
print("size:", round(Path("waste-v1.pt").stat().st_size / 1e6, 1), "MB")

from google.colab import files
files.download("waste-v1.pt")

## 10 · Integrate — the two lines

Put `waste-v1.pt` in `models/`, then edit `config/config.yaml`:

```yaml
detection:
  model_path: "models/waste-v1.pt"      # was models/yolov8n.pt
  label_map:  "config/labels_waste.yaml" # was config/labels_coco.yaml
```

Then check nothing above the detector needed touching:

```bash
python -m tests.test_labels      # the agents, against the new vocabulary
python -m tests.test_pipeline
python main.py                   # hold a battery up to the camera
```

### Then get your results chapter

With a model that can see waste, the evaluation harness finally has something to
measure:

```bash
python -m evaluation.capture --label battery      # ~20 frames per class
python -m evaluation.evaluate
```

Paste the resulting accuracy, wrong-bin rate and confusion matrix into your README.
That table is the thing your project has been missing — it turns "the system is
correct" into "the system works, and here is the number."
